# 02 — Grid Search

Grid Search je najjednostavnija metoda optimizacije hiperparametara: isprobamo sve moguće kombinacije vrednosti sa unapred zadate mreže (grid) i uzmemo onu koja daje najbolji rezultat na unakrsnoj validaciji.

Radimo za dva modela (SVM i Random Forest) i dva skupa podataka (Breast Cancer i Digits), znači ukupno 4 kombinacije.

**Napomena o metodologiji:** kombinujemo dva pristupa. Pretragu hiperparametara radimo preko **3-fold unakrsne validacije** direktno na trening delu (80%), ponovljeno **10 puta sa različitim random seed-ovima** — ovo prati metodologiju iz referentnog rada ([Yang & Shami, 2020](https://www.eng.uwo.ca/oc2/publications/thepublicationpdfs/2020_YangShami_NeuroComputing.pdf), poglavlje 7.1) i daje nam mean/std tačnosti (koliko je rezultat dobar i koliko stabilan).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
import time

## Učitavanje podataka

In [2]:
breast_cancer = load_breast_cancer()
digits = load_digits()

X_bc, y_bc = breast_cancer.data, breast_cancer.target
X_dg, y_dg = digits.data, digits.target

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc)
X_dg_train, X_dg_test, y_dg_train, y_dg_test = train_test_split(
    X_dg, y_dg, test_size=0.2, random_state=42, stratify=y_dg)

skupovi_podataka = {
    'breast_cancer': (X_bc_train, y_bc_train),
    'digits': (X_dg_train, y_dg_train),
}

#10 razlicitih random seed-ova
SEEDOVI = list(range(10))
BROJ_FOLDOVA = 3  #isto kao u referentnom radu

## Definicija grida

Za SVM biramo `C` (koliko strogo kažnjavamo greške) i `gamma` (koliko "lokalno" reaguje RBF kernel). Za Random Forest biramo `n_estimators` (broj stabala), `max_depth` (maksimalna dubina stabla) i `min_samples_split` (minimalan broj uzoraka da bi se čvor dalje delio).

In [3]:
param_grid_svm = {
    'C': [0.01, 0.1, 1, 10, 100],
    'gamma': [0.0001, 0.001, 0.01, 0.1, 1],
}

param_grid_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
}

broj_kombinacija_svm = len(param_grid_svm['C']) * len(param_grid_svm['gamma'])
broj_kombinacija_rf = (len(param_grid_rf['n_estimators'])
                       * len(param_grid_rf['max_depth'])
                       * len(param_grid_rf['min_samples_split']))

print('SVM broj kombinacija:', broj_kombinacija_svm)
print('RF broj kombinacija: ', broj_kombinacija_rf)

SVM broj kombinacija: 25
RF broj kombinacija:  60


## Pokretanje

Za svaku kombinaciju pokrećemo Grid Search 10 puta — svaki put sa drugačijom podelom na foldove (`KFold` sa drugim `random_state`) — i pamtimo najbolji rezultat svakog pokretanja. Na kraju računamo prosek i standardnu devijaciju.

In [4]:
rezultati_grid = [] 

for ime_skupa, (X, y) in skupovi_podataka.items():
    for ime_modela in ['svm', 'random_forest']:

        skorovi_po_seedu = []     # best_score iz svakog od 10 pokretanja
        vremena_po_seedu = []     # trajanje svakog pokretanja
        parametri_po_seedu = []   # best_params iz svakog pokretanja

        for seed in SEEDOVI:
            kf = KFold(n_splits=BROJ_FOLDOVA, shuffle=True, random_state=seed)

            if ime_modela == 'svm':
                model = SVC(kernel='rbf', random_state=42)
                grid = param_grid_svm
            else:
                model = RandomForestClassifier(random_state=seed, n_jobs=-1)
                grid = param_grid_rf

            pocetak = time.time()
            grid_search = GridSearchCV(model, grid, cv=kf, scoring='accuracy', n_jobs=-1)
            grid_search.fit(X, y)
            trajanje = time.time() - pocetak

            skorovi_po_seedu.append(grid_search.best_score_)
            vremena_po_seedu.append(trajanje)
            parametri_po_seedu.append(grid_search.best_params_)

        #uzimamo parametre iz seed-a koji je dao najbolji rezultat
        indeks_najboljeg = int(np.argmax(skorovi_po_seedu))

        rezultati_grid.append({
            'dataset': ime_skupa,
            'model': ime_modela,
            'method': 'grid_search',
            'mean_score': np.mean(skorovi_po_seedu),
            'std_score': np.std(skorovi_po_seedu),
            'n_evaluations': broj_kombinacija_svm if ime_modela == 'svm' else broj_kombinacija_rf,
            'mean_time_sec': np.mean(vremena_po_seedu),
            'best_params': parametri_po_seedu[indeks_najboljeg],
        })

        print(f'{ime_skupa:15s} {ime_modela:15s} '
              f'mean_score={np.mean(skorovi_po_seedu):.4f} (+/- {np.std(skorovi_po_seedu):.4f})  '
              f'mean_time={np.mean(vremena_po_seedu):6.1f}s')

breast_cancer   svm             mean_score=0.9446 (+/- 0.0050)  mean_time=   0.3s
breast_cancer   random_forest   mean_score=0.9620 (+/- 0.0024)  mean_time=  14.2s
digits          svm             mean_score=0.9908 (+/- 0.0021)  mean_time=   1.8s
digits          random_forest   mean_score=0.9750 (+/- 0.0016)  mean_time=  18.7s


## Rezultati

In [5]:
df_grid = pd.DataFrame(rezultati_grid)
df_grid

,dataset,model,method,mean_score,std_score,n_evaluations,mean_time_sec,best_params
0,breast_cancer,svm,grid_search,0.944606,0.004981,25,0.310282,"{'C': 1, 'gamma': 0.0001}"
1,breast_cancer,random_forest,grid_search,0.961971,0.002412,60,14.186125,"{'max_depth': 10, 'min_samples_split': 2, 'n_e..."
2,digits,svm,grid_search,0.990814,0.002106,25,1.819838,"{'C': 1, 'gamma': 0.001}"
3,digits,random_forest,grid_search,0.975017,0.001570,60,18.726578,"{'max_depth': 10, 'min_samples_split': 2, 'n_e..."


In [8]:
kolone_rezultata = ['dataset', 'model', 'method', 'mean_score', 'std_score',
                    'n_evaluations', 'mean_time_sec', 'best_params']

import os
os.makedirs('./results', exist_ok=True)

df_grid[kolone_rezultata].to_csv('./results/all_results.csv', mode='a', header=False, index=False)
print('Rezultati sačuvani u results/all_results.csv')

Rezultati sačuvani u results/all_results.csv
